In [18]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mc
from scipy import stats
import statsmodels.stats.multitest as multitest
import warnings

warnings.filterwarnings('ignore')

# =============================================================================
# Settings
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
DPI_SETTING = 600

# Force math text to support bold-italic properly
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Arial:italic'
plt.rcParams['mathtext.bf'] = 'Arial:bold'
plt.rcParams['mathtext.bfit'] = 'Arial:italic:bold'

def lighten_hex(hex_color, factor=0.3):
    r, g, b = mc.to_rgb(hex_color)
    return mc.to_hex((r + (1-r)*factor, g + (1-g)*factor, b + (1-b)*factor))

# RS is DarkTurquoise, Erythritol is Red
color_rs = lighten_hex('#00CED1', 0.3)
color_erythritol = lighten_hex('#FF0000', 0.3)

def clean_and_convert_to_nan(vals):
    s_vals = pd.Series(vals).astype(str).str.strip()
    s_vals = s_vals.str.replace(r'\.E\+', 'E+', regex=True)
    s_vals = s_vals.str.replace(r'\.E\-', 'E-', regex=True)
    s_vals = s_vals.replace(['Undetermined', '-', 'nan', 'NaN', '#VALUE!', '', 'None'], np.nan)
    return pd.to_numeric(s_vals, errors='coerce')

# =============================================================================
# Plotting Function
# =============================================================================
def plot_scatter_qval(df_plot, x_col, y_col, x_label, y_label, color_hex, text_position, q_val, out_filename):
    if len(df_plot) < 3:
        print(f"Skipping {out_filename}: Not enough data.")
        return

    r_val, _ = stats.pearsonr(df_plot[x_col], df_plot[y_col])

    fig, ax = plt.subplots(figsize=(5, 5))

    # Line styling (gray solid line for lack of meaningful correlation r < 0.4)
    if abs(r_val) >= 0.4:
        line_style = {'linewidth': 2, 'color': 'black', 'alpha': 1.0, 'linestyle': '-'}
    else:
        line_style = {'linewidth': 2, 'color': 'gray', 'alpha': 0.5, 'linestyle': '-'}

    sns.regplot(x=x_col, y=y_col, data=df_plot, ax=ax, color=color_hex,
                scatter_kws={'s': 60, 'alpha': 0.7, 'edgecolors': 'white', 'linewidths': 0.5},
                line_kws=line_style)

    ax.axhline(0, color='black', linestyle=':', linewidth=1.5)
    if df_plot[x_col].min() < 0:
        ax.axvline(0, color='black', linestyle=':', linewidth=1.5)

    ax.set_ylabel(y_label, fontsize=14, fontweight='bold', labelpad=10)
    ax.set_xlabel(x_label, fontsize=14, fontweight='bold', labelpad=10)

    ax.tick_params(axis='both', labelsize=12, width=1.5, length=5)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_box_aspect(1)

    q_text = "q < 0.001" if q_val < 0.001 else f"q = {q_val:.3f}"
    stats_text = f"$r = {r_val:.2f}$\n{q_text}"

    text_bbox = dict(facecolor='white', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2')

    if text_position == 'top_right':
        x_pos, y_pos, va, ha = 0.95, 0.95, 'top', 'right'
    elif text_position == 'bottom_right':
        x_pos, y_pos, va, ha = 0.95, 0.05, 'bottom', 'right'
    else:
        x_pos, y_pos, va, ha = 0.05, 0.95, 'top', 'left'

    ax.text(x_pos, y_pos, stats_text, transform=ax.transAxes,
            fontsize=14, fontweight='normal', va=va, ha=ha, color='black', bbox=text_bbox)

    plt.tight_layout()
    plt.savefig(out_filename, dpi=DPI_SETTING, bbox_inches='tight', transparent=True)
    plt.close()
    print(f"Generated: {out_filename}")

# =============================================================================
# Data Loading & Processing
# =============================================================================
df_butyrate = pd.read_csv('Butyrate(mM).csv')
donor_cols = [c for c in df_butyrate.columns if c.startswith('HS-')]
y_label_butyrate = r'$\Delta$ Butyrate (mM)'

# --- 1. RS: Delta Butyricicoccus (qPCR) vs Delta Butyrate ---
try:
    df_tax_rs = pd.read_csv('Butyricicoccus(qPCR).csv')
except UnicodeDecodeError:
    df_tax_rs = pd.read_csv('Butyricicoccus(qPCR).csv', encoding='shift_jis')

ctrl_tax_rs = clean_and_convert_to_nan(df_tax_rs[df_tax_rs['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
sub_tax_rs = clean_and_convert_to_nan(df_tax_rs[df_tax_rs['KULFFI'].str.strip() == 'Resistant starch'][donor_cols].iloc[0])
delta_tax_rs = np.log10(sub_tax_rs + 1) - np.log10(ctrl_tax_rs + 1)

ctrl_buty_rs = clean_and_convert_to_nan(df_butyrate[df_butyrate['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
sub_buty_rs = clean_and_convert_to_nan(df_butyrate[df_butyrate['KULFFI'].str.strip() == 'Resistant starch'][donor_cols].iloc[0])
delta_buty_rs = sub_buty_rs - ctrl_buty_rs

df_rs = pd.DataFrame({'tax': delta_tax_rs.values, 'buty': delta_buty_rs.values}).dropna()

# --- 2. Erythritol: Delta Anaerostipes (qPCR) vs Delta Butyrate ---
try:
    df_tax_ery = pd.read_csv('A.caccae.csv')
except UnicodeDecodeError:
    df_tax_ery = pd.read_csv('A.caccae.csv', encoding='shift_jis')

ctrl_tax_ery = clean_and_convert_to_nan(df_tax_ery[df_tax_ery['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
sub_tax_ery = clean_and_convert_to_nan(df_tax_ery[df_tax_ery['KULFFI'].str.strip() == 'Erythritol'][donor_cols].iloc[0])
delta_tax_ery = np.log10(sub_tax_ery + 1) - np.log10(ctrl_tax_ery + 1)

ctrl_buty_e = clean_and_convert_to_nan(df_butyrate[df_butyrate['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0])
sub_buty_e = clean_and_convert_to_nan(df_butyrate[df_butyrate['KULFFI'].str.strip() == 'Erythritol'][donor_cols].iloc[0])
delta_buty_e = sub_buty_e - ctrl_buty_e

df_ery = pd.DataFrame({'tax': delta_tax_ery.values, 'buty': delta_buty_e.values}).dropna()

# =============================================================================
# FDR Correction (for the 2 independent correlations)
# =============================================================================
p_vals = []
for df in [df_rs, df_ery]:
    if len(df) >= 3:
        _, p = stats.pearsonr(df['tax'], df['buty'])
        p_vals.append(p)
    else:
        p_vals.append(1.0)

_, q_vals, _, _ = multitest.multipletests(p_vals, alpha=0.05, method='fdr_bh')
q_rs, q_ery = q_vals

# =============================================================================
# Generating Figure 5f (RS) and 5g (Erythritol)
# =============================================================================

# 5f: RS (Top Right text)
x_label_rs = r'$\Delta$ $\mathbfit{Butyricicoccus}$ spp.' + '\n' + r'(Log$_{\mathbf{10}}$ copies/mL)'
plot_scatter_qval(df_rs, 'tax', 'buty', x_label_rs, y_label_butyrate, color_rs, 'top_right', q_rs, 'Figure_5f.pdf')

# 5g: Erythritol (Bottom Right text, Red color)
x_label_ery = r'$\Delta$ $\mathbfit{Anaerostipes\ caccae}$' + '\n' + r'(Log$_{\mathbf{10}}$ copies/mL)'
plot_scatter_qval(df_ery, 'tax', 'buty', x_label_ery, y_label_butyrate, color_erythritol, 'bottom_right', q_ery, 'Figure_5g.pdf')

Generated: Figure_5f.pdf
Generated: Figure_5g.pdf
